In [1]:
from dotenv import load_dotenv



load_dotenv(override=True)

True

In [2]:
from langchain.tools import tool

@tool
def getWeather(location: str) -> str:
    """
    Get the current weather in a given location
    Args:
        location (str): The location to get the weather for
    """
    return f"The current weather in {location} is sunny."

In [3]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model("openai:qwen-max", temperature=0, base_url=os.getenv("ROOT_URL"))
print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


In [9]:
stream = model.stream("who are you?")

for chunk in stream:
    print(chunk.content, end="")

I'm Qwen, a large language model created by Alibaba Cloud. I'm designed to assist with a wide range of tasks, from answering questions and providing information to helping with creative writing, coding, and more. How can I assist you today?

In [19]:
from pydantic import BaseModel, Field
from typing import Literal


class WeatherInput(BaseModel):
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(description="Temperature unit preference", default="celsius")
    include_forecast: bool = Field(description="Include 5-day forecast", default=False)


In [22]:
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast"""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degress {units[0].upper()}"
    if include_forecast:
        result += "/nNext 5 days: Sunny"
    return result 

In [23]:
get_weather.invoke({"location": "hk", "include_forecast": False})

'Current weather in hk: 22 degress C'

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
)


agent = create_agent(
    model = model,
    tools = [tavily_search_tool]
)


for token, meta in agent.stream(
    {
        "messages": HumanMessage(content="How is HK weather tmr")
    },
    stream_mode = "messages"
):
    if token:
        print(token.content, end="", flush=True)

Current weather in Hong Kong: 22 degress C/nNext 5 days: SunnyThe weather in Hong Kong tomorrow is forecasted to be sunny with a temperature of 22 degrees Celsius.

In [6]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.runnables import RunnableConfig

with SqliteSaver.from_conn_string("checkpoints.sqlite") as checkpointer:
    checkpointer.setup()
    agent = create_agent(
        model=model,
        tools=[getWeather],
        middleware=[
            SummarizationMiddleware(
                model=model,
                trigger=("tokens", 4000),
                keep=("messages", 20)
            )
        ],
        checkpointer=checkpointer,
    )

    config: RunnableConfig = {"configurable": {"thread_id": "1"}}
    agent.invoke({"messages": "hi, my name is bob"}, config)
    agent.invoke({"messages": "write a short poem about cats"}, config)
    agent.invoke({"messages": "now do the same but for dogs"}, config)
    final_response = agent.invoke({"messages": "what's my name?"}, config)






In [9]:
for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

hi, my name is bob
================================== Ai Message ==================================

Hello Bob! It's nice to meet you. How can I assist you today?
================================ Human Message =================================

write a short poem about cats
================================== Ai Message ==================================

In the quiet corners, they softly tread,
Silent paws that barely make a thread.
Eyes like lanterns in the night,
Cats, with fur as soft as light.

Whiskers twitching, tails held high,
Underneath the moon's watchful eye.
Purring secrets, warm and sweet,
Guardians of mysteries, oh so discrete.

Nimble dancers on the sill,
Chasing shadows with a thrill.
Graceful, noble, yet so quaint,
In the realm where dreams acquaint.
================================ Human Message =================================

now do the same but for dogs
=============================